# Lab — Correlation & Regression

**Solution notebook**

This notebook demonstrates one possible approach. Your exact numbers may vary depending on your choices (e.g., features selected, filtering).

In [ ]:
# Imports
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import statsmodels.api as sm
from scipy import stats

# Display options (optional)
pd.set_option("display.max_columns", 50)

In [ ]:
from pathlib import Path
import pandas as pd

# Load data using a project-root anchored path
PROJECT_ROOT = Path.cwd().parent
data_path = PROJECT_ROOT / "data" / "validated_dataset_m11.csv"

df = pd.read_csv(data_path)
df.head()


In [ ]:
# Quick sanity checks
df.shape, df.dtypes

In [ ]:
# If there are any obvious missing values, decide how to handle them.
# (This dataset should already be validated, but we still check.)
df.isna().sum().sort_values(ascending=False).head(10)

## 1) Choose variables

We'll analyze a **numeric outcome** and one or more **numeric predictors**.

In this sample solution, we’ll use:
- `order_total_usd` as the outcome (Y)
- `items_count` as the main predictor (X)

If your dataset uses slightly different column names, update the cells below accordingly.

In [ ]:
# Identify numeric columns
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
numeric_cols

In [ ]:
# Choose columns for correlation/regression
# This dataset tracks revenue and units sold.
y_col = "revenue_usd"
x_col = "units_sold"

# Basic guardrails
assert y_col in df.columns, f"Missing column: {y_col}. Available: {list(df.columns)}"
assert x_col in df.columns, f"Missing column: {x_col}. Available: {list(df.columns)}"

df[[y_col, x_col]].describe()


## 2) Correlation

We compute a correlation coefficient and interpret direction/strength.

- Pearson: linear relationship (requires numeric, roughly continuous)
- Spearman: rank-based (robust to non-linear monotonic relationships)

In [ ]:
from scipy import stats

# Drop rows with missing values in selected columns
corr_df = df[[y_col, x_col]].dropna()

pearson_r, pearson_p = stats.pearsonr(corr_df[x_col], corr_df[y_col])
spearman_r, spearman_p = stats.spearmanr(corr_df[x_col], corr_df[y_col])

print(f"Pearson r:  {pearson_r:.3f} (p={pearson_p:.4g})")
print(f"Spearman r: {spearman_r:.3f} (p={spearman_p:.4g})")


**Interpretation (example):**
- Pearson r indicates the *direction* (+/-) and *strength* (0 to 1) of a linear relationship.
- p-value indicates whether the correlation is statistically distinguishable from 0 (given assumptions).

Always interpret with context: correlation ≠ causation.

In [ ]:
# Optional: correlation matrix for several numeric columns
df[numeric_cols].corr(numeric_only=True).round(2)

## 3) Visual check

A scatter plot helps confirm whether a linear model is reasonable.

In [ ]:
plt.figure(figsize=(7,5))
plt.scatter(corr_df[x_col], corr_df[y_col], alpha=0.6)
plt.title(f"{y_col} vs {x_col}")
plt.xlabel(x_col)
plt.ylabel(y_col)
plt.show()

## 4) Simple Linear Regression (StatsModels)

We fit:  
\( Y = \beta_0 + \beta_1 X + \epsilon \)

In [ ]:
# Prepare X and y
X = corr_df[[x_col]]
X = sm.add_constant(X)  # adds intercept
y = corr_df[y_col]

model = sm.OLS(y, X).fit()
model.summary()

### How to interpret key outputs
- **coef** for `items_count`: expected change in `order_total_usd` for a +1 change in items_count (holding others constant; here there are no other predictors).
- **P>|t|**: significance of that coefficient.
- **R-squared**: fraction of variance explained by the model (not “accuracy”).
- **Confidence intervals**: plausible range for the true coefficient.

In [ ]:
# Predicted values and residuals
corr_df = corr_df.copy()
corr_df["predicted"] = model.predict(X)
corr_df["residual"] = corr_df[y_col] - corr_df["predicted"]

corr_df[[x_col, y_col, "predicted", "residual"]].head()

In [ ]:
# Residual plot (quick diagnostic)
plt.figure(figsize=(7,5))
plt.scatter(corr_df["predicted"], corr_df["residual"], alpha=0.6)
plt.axhline(0, linestyle="--")
plt.title("Residuals vs Predicted")
plt.xlabel("Predicted")
plt.ylabel("Residual")
plt.show()

## 5) Communicate findings (non-technical)

Write a short summary that a stakeholder can understand.

Example template:
- What relationship did you test?
- What did you find (direction + size)?
- How confident are you (mention p-value / CI in plain language)?
- What are limitations?

**Example summary (edit with your computed values):**

We examined whether orders with more items tend to have higher total value.  
We found a positive relationship: as `items_count` increases, `order_total_usd` tends to increase.  
A simple linear regression suggests that adding one item is associated with an average increase in order total (see model coefficient), and the effect is statistically significant based on the p-value reported in the regression output.  

Limitations: This is observational data. Correlation and regression do not prove causation, and other factors (e.g., discounts, customer segment, product mix) may explain part of the relationship.